# Notebook consacré à l'analyse des questions relatives à la Science Ouverte

In [ ]:
import pandas as pd

import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket
import scipy.stats

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")


df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")
list_affil = pd.read_csv("list_affiliation.csv", sep =",")
df0 = df0.loc[~df0.q45_clé.isin(["9EAP-NB4B","QNYZ-3MH2"])].merge(list_affil, on = ["q45_clé", "q44_ufr_labo"], how = "left")

In [ ]:
list_nominal_simple = [x for x in df_col.label.loc[(df_col.type.isin(["simple_nominal", "booléen","ordinal"]))]]


In [ ]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df_split.apply(lambda row: row[column].replace(";","|") ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).sort_values("nb", ascending=False).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100
    
    return df_explode, gb_data
    

In [ ]:
def grouped_question(data, column, index = "q45_clé"):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_tmp = data.copy()
    gb_data = df_tmp.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data

In [ ]:
df0

In [ ]:
df_col["question_family"] = df_col.label.apply(lambda row : row.split("_")[0])

dict_question = dict(zip(df_col.label.loc[df_col.label.isin(list_nominal_simple)], df_col.question_family.loc[df_col.label.isin(list_nominal_simple)]))


# Connaissance et pratique de la Science Ouverte
Plusieurs questions de l'enquête portent sur les connaissances et les pratiques de la Science Ouverte. La première question demande ainsi aux participants s'ils "sont "familiers des principes de la Science Ouverte". 57% ont répond "oui, un peu", 32 "oui, tout à fait" et 10% "non". Ils sont par ailleurs 40% à avoir déposé plusieurs fois des publications sur Hal, et 35% le font systématiquement. 

| Principes de la science ouverte   |   nb |    freq | 
|:-------------------|-----:|--------:|
| Non                |   12 |    10.3 | 
| Oui, tout à fait   |   38 |    32.5 |
| Oui, un peu        |   67 |    57.3 |
|                    |  117 |   100.0 |


| q2_hal_depot          |   nb |   total |    freq |   total_freq |
|:----------------------|-----:|--------:|--------:|-------------:|
| Non                   |   15 |     117 | 12.8205 |          100 |
| Oui, plusieurs fois   |   47 |     117 | 40.1709 |          100 |
| Oui, systématiquement |   41 |     117 | 35.0427 |          100 |
| Oui, une fois         |   14 |     117 | 11.9658 |          100 |

|| q4_diff_data            |   nb |   total |      freq |  
|---:|:------------------------|-----:|--------:|----------:|
| Autre, précisez         |   25 |     117 | 21.3675   |   
| Non                     |   84 |     117 | 71.7949   |  
| Oui, sur Data Paris 8   |    1 |     117 |  0.854701 |  
| Oui, sur Nakala         |    3 |     117 |  2.5641   |   
| Oui, sur Zenodo, Nakala |    4 |     117 |  3.4188   | 

En ce qui concerne l'ouverture des données, 72% des personnes interrogées ne l'ont jamais fait. ParmiScience les 28% personnes restantes, 6% d'entre elles ont utilisé au moins une fois des entrepôts généralistes comme Data Paris 8 et Nakala ont été utilisés. parmi les autres moyens de diffusion utilisés, 10 personnes utilisent Open Science Framework (OSF), 6 mettent leurs données sur le site web personnel et 3 les entreposent sur un dépôt "git". On observe également que 3 répondent déposent leurs données sur les entrepots spécialisés, liés à la linguistiques, Ortolang et Cocoon.

![](viz/autre_entrepot_rec.png)

Enfin, un peu moins de 10% des personnes interrogées ont déjà déposé des productions sur Octaviana. Ces productions sont des captations d'événements pour trois d'entre elles et des travaux universitaires et des publications pour les huit autres. Par ailleurs seules 9 personnes sur les 91 n'ayant jamais déposé de production sur Octaviana ont précisé qu'elles ne connaissaient pas la bibliothèque numérique. En tout une personne sur cinq n'en a jamais entendu parlé.


In [ ]:
so_principe = grouped_question(df0, column="q1_so_principles", index = "q45_clé")
print(so_principe)
depot_hal = grouped_question(df0, column="q2_hal_depot", index = "q45_clé")
print("\n### Dépôt dans hal\n", depot_hal)
octaviana = grouped_question(df0, column="q3_octavi_rec", index = "q45_clé")
print("\n### Dépôt dans Octaviana\n", octaviana)

df_exp, octaviana2 = split_multiple_choices(df0, column="q3_octavi_depot", index="q45_clé", sep = '|')
print("\n### Dépôt dans Octaviana détaillé \n", octaviana2.to_markdown())



In [ ]:
df0.loc[df0.q4_diff_data !="Non", "q4_diff_data_grouped"] = "Oui"
df0.loc[df0.q4_diff_data =="Non", "q4_diff_data_grouped"] = "Non"

depot_donnee = grouped_question(df0, column="q4_diff_data_grouped", index = "q45_clé")
print("\n### Dépôt des données de recherche\n", depot_donnee)

df_exp, depot_donnee2 = split_multiple_choices(df0, column="q4_diff_data", index="q45_clé", sep = '|')
print("\n### Dépôt dans Octaviana détaillé \n", depot_donnee2.to_markdown(index=False))

## Les autres modes de dépôt des données de recherche

In [ ]:
df0.loc[df0["q4_diff_data"]=='Autre, précisez', "q4_diff_data_rec"] = df0.q4_autres_entrepots_rec
df0.loc[df0["q4_diff_data"]!='Autre, précisez', "q4_diff_data_rec"] = df0.q4_diff_data
df0.loc[df0["q4_diff_data_rec"].isna(), "q4_diff_data_rec"] = "NSP"
q4 = split_multiple_choices(df0, column="q4_diff_data_rec", index="q45_clé", sep = '|')

In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(1, figsize=(6,4))

# Plot the total crashes
sns.set_color_codes("pastel")
q4_other = df0.loc[df0.q4_diff_data=="Autre, précisez"].fillna("Non précisé")
col= "q4_autres_entrepots_rec"
df_exp, gb_data = split_multiple_choices(q4_other, column=col, index="q45_clé", sep = '|')
sns.barplot(x="total", y=col, data=gb_data,
            label="Non", color="b", ax=ax)
sns.barplot(x="nb", y=col, data=gb_data,
            label="Oui", color="r", ax=ax)
titre = "Les autres lieux de dépot des données de recherche"
ax.yaxis.set_label_text("")
ax.xaxis.set_label_text("")
ax.set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
plt.savefig(f"viz/autre_entrepot_rec.png", bbox_inches='tight', dpi = 200)

## Développement et dépôt de codes ou de logiciels

In [ ]:
dev_code= grouped_question(df0, column="q6_dev_code", index = "q45_clé")
print(dev_code)

df_exp, sw_depot2 = split_multiple_choices(df0, column="q7_software_depot", index="q45_clé", sep = '|')
print("\n### Dépôt dans Octaviana détaillé \n", sw_depot2.to_markdown())

Près d'un quart des répondants ont déjà été amenés à développer ou faire développer du code ou des logiciels dans le cadre de leurs recherches, dont deux n'ont jamais déposé leur code ou logiciel sur un entrepot dédié.

Si le dépôt de code sur Github et Gitlab restent les solutions les plus usitées, 6 personnes déposent leus codes et logiciels ailleurs : 
* "Sur les dépôts SSC des commandes Stata";
* "sur des repositories personnels";
* "sur Hal";
* "sur Ortolang et Huma-Num",
* "Sur des instances auto-hébergées de Gitlab"
* ou "sur l'INPI".



## Les besoins d'accompagnement à la Science Ouverte

Mis à part l'accompagnement au dépot dans Hal, une majorité de répondants juge les accompagnement à la Science Ouverte comme étant utiles. Les licences de diffusion des résultats de recherche est le sujet pour lequel les répondants ont manifesté le plus d'intérêt : 87% considèrent qu'un accompagnement sur ce thème serait "plutôt utile" (52%) voire "très utile" (35%). Vient ensuite la "fairisation" des données de recherche avec 81% des participants à l'enquêtre déclarant qu'il serait utile d'être accompagner dan sle processus permettant de produire des données respectant les principes FAIR : Facile à trouver, Accessible, Interopérable, et Réutilisable. De façon général, les accompagnements apparaissent d'autant plus utiles qu'ils portent sur des aspects juridiques (licence, RGPD) ou des sujets liés à l'ouverture des données (métadonnées, dépots, principes FAIR).



In [ ]:
for col in df_col.label.loc[(df_col.group=="6_help_SO")&(df_col.type=="ordinal")]:
    gb_data= grouped_question(df0, column=col, index="q45_clé")
    print(gb_data.to_markdown())

In [ ]:
[col for col in df_col.label.loc[(df_col.group=="6_help_SO")&(df_col.type=="ordinal")]]

In [ ]:
dict_accomp = {'q8_depot_hal_help':"Dépot sur Hal",
 'q8_rediger_pdg_help': 'Rédiger un PGD',
 'q8_respect_rgpd_help': 'Respecter le RGPD',
 'q8_licences_help': 'Choix des licences de diffusion',
 'q8_depot_donnees_help': 'Déposer des données',
 'q8_metadonnees_help': 'Connaître les standards de métadonnées',
 'q8_data_paper_help':'Rédiger un data paper',
 'q8_fair_data_help':'Fairiser les données'
              }

In [ ]:

i=-1
for col in df_col.label.loc[(df_col.group=="6_help_SO")&(df_col.type=="ordinal")]:
    i+=1
    dftmp = df0[["q45_clé", col]].rename(columns={col:"interet_accomp"})
    dftmp["accomp_name"]= dict_accomp[col]


    if i == 0:
        q8 = dftmp.copy()
    else:
        q8 = pd.concat([q8, dftmp])


In [ ]:
q8_dis = pd.crosstab(q8.accomp_name, q8.interet_accomp, normalize="index").cumsum(axis=1).stack().reset_index(name='nb').sort_values(by=["accomp_name","interet_accomp"],
                                                                                                                                     ascending=[True, False])


In [ ]:
fig, ax = plt.subplots(1, figsize=(6,4))

sns.set_theme(style="ticks", context="paper")

   
g = sns.barplot(x="nb", y="accomp_name", data=q8_dis, hue="interet_accomp", dodge=False)

titre = "L'utilité des accompagnements à la science ouverte"
ax.yaxis.set_label_text("")
ax.xaxis.set_label_text("")
ax.set_xticks(ticks = [x/10 for x in range(11)])
ax.set_title(titre.replace("<i>","(").replace("</i>",")"))
ax.legend(bbox_to_anchor=(1.05, 1),
                         loc='upper left', borderaxespad=0.)

sns.despine(left=True, bottom=True)
plt.savefig(f"viz/util_accomp_so.png", bbox_inches='tight', dpi = 200)

In [ ]:
fig, ax = plt.subplots(1, figsize=(6,4))

sns.set_color_codes("pastel")

q8_grp = q8.groupby(["accomp_name","interet_accomp"]).agg(nb=("q45_clé", "size")).sort_values(by=["accomp_name","nb"], ascending=[True, False])

    
sns.barplot(x="nb", y="accomp_name", data=q8_grp, hue="interet_accomp", dodge=True)
titre = "L'utilité des accompagnements à la science ouverte"
ax.yaxis.set_label_text("")
ax.xaxis.set_label_text("")
ax.set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
#plt.savefig(f"viz/autre_entrepot_rec.png", bbox_inches='tight', dpi = 200)

# Types de données

In [ ]:
#%%capture cap
for n, family in enumerate(df_col.question_family.unique()):
    list_quest = [x for x in dict_question if dict_question[x]==family]
    if len(list_quest) > 0:
        question = df_col.question.loc[df_col.question_family == family].iloc[0]
        print("==================\n", question)
        for m, col in enumerate(list_quest):
            print(df_col.name.loc[df_col.label==col].iloc[0])
            gb_data = grouped_question(df0, column=col, index = "q45_clé")
            print(gb_data)
            print("---------------------\n")
            
        


In [ ]:
with open("tableau_frequence_question_simple.txt", 'w') as file_in:
    file_in.write(cap.stdout)

In [ ]:
def tab_croise(data, x, y, regroup_y = True, khideux = False) :
    """
    x = variable en ligne
    y = variable en colonne
    """
    if regroup_y == True:

        data.loc[data[y].str.lower().str.contains("oui"), f"{y}_rec"] = "Oui"
        data.loc[data[y].str.lower().str.contains("non"), f"{y}_rec"] = "Non"

        cross_tab = pd.crosstab(data[x], data[f"{y}_rec"], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[f"{y}_rec"], margins = False)
    else:
        cross_tab = pd.crosstab(data[x], data[y], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[f"{y}_rec"], margins = False)

    if khideux == True:
        print(scipy.stats.chi2_contingency(cross_tab0))
        st_chi2, st_p, st_dof, st_exp = scipy.stats.chi2_contingency(cross_tab0)
        chi2 = pd.DataFrame(data={"stats":["chi2","df","p-value"], "values":[st_chi2, st_dof, st_p]})
        cross_tab = pd.concat([cross_tab, chi2])
    else:
        pass

    return cross_tab.fillna("").reset_index()

In [ ]:
tab_croise(df0, x= "q25_statut_rec", y="q2_hal_depot", regroup_y = True, khideux = True)

In [ ]:
(15*107)/119

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
list_family = [x for x in dict_question.values()]
list_family_dedup = []
for x in list_family:
    if list_family_dedup.count(x) == 0:
        list_family_dedup.append(x)
    else:
        pass


In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(len(list_family), figsize=(10, 30))

# Plot the total crashes
sns.set_color_codes("pastel")

for n, family in enumerate(df_col.question_family.unique()):
    list_quest = [x for x in dict_question if dict_question[x]==family]
    if len(list_quest) > 0:
        for m, col in enumerate(list_quest):
            #print(col)
            gb_data = grouped_question(df0, column=col, index = "q45_clé")
        
            sns.barplot(x="freq", hue=col, data=gb_data, ax=ax[n])

       

In [ ]:
sns.barplot(x="nb", y=col, data=gb_data,
                label="effectif", color="r", ax=ax[n])